In [3]:
# train_mt_embed_reg.py

import os
import math
import json
import random
from dataclasses import dataclass
from typing import Dict, List, Any, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

# Optional metrics
from comet import download_model, load_from_checkpoint
from bert_score import score as bertscore_score

In [4]:
MODEL_NAME = "Helsinki-NLP/opus-mt-es-en"
SRC_LANG = "es"
TGT_LANG = "en"

In [13]:
# -------------------------
# 1) Data
# -------------------------
# def load_translation_data():
#     """
#     Spanish-English dataset.
#     Examples:
#       - IWSLT Spanish-English
#       - OPUS/OpenSubtitles/TED-style data
#       - your own parallel corpus
#     """
#     ds = load_dataset("...", split={...})
#     return ds["train"], ds["validation"], ds["test"]

def load_translation_data(seed=42, test_size=0.1, val_size=0.1):
    """
    Spanish-English dataset for a first experiment.
    OPUS Books has only a train split, so we create val/test manually.
    """
    ds = load_dataset("Helsinki-NLP/opus_books", "en-es")

    # 80/10/10-style split by default
    tmp = ds["train"].train_test_split(test_size=(test_size + val_size), seed=seed)
    train_ds = tmp["train"]

    tmp2 = tmp["test"].train_test_split(
        test_size=val_size / (test_size + val_size),
        seed=seed,
    )
    val_ds = tmp2["train"]
    test_ds = tmp2["test"]

    # return ds["train"], ds["validation"], ds["test"]
    return train_ds, val_ds, test_ds

def tokenize_batch(examples, tokenizer, max_source_len=128, max_target_len=128):
    sources = [ex["translation"][SRC_LANG] for ex in examples]
    targets = [ex["translation"][TGT_LANG] for ex in examples]

    model_inputs = tokenizer(
        sources,
        max_length=max_source_len,
        truncation=True,
        padding=False,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_len,
            truncation=True,
            padding=False,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_source_len=128, max_target_len=128):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        ex = self.dataset[idx]
        src = ex["translation"][SRC_LANG]
        tgt = ex["translation"][TGT_LANG]

        model_inputs = self.tokenizer(
            src,
            max_length=self.max_source_len,
            truncation=True,
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                tgt,
                max_length=self.max_target_len,
                truncation=True,
            )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

In [14]:
# -------------------------
# 2) Embedding-aware loss
# -------------------------
def shift_right(labels: torch.Tensor, pad_token_id: int, decoder_start_token_id: int):
    """
    Create decoder input ids from labels.
    """
    shifted = labels.new_zeros(labels.shape)
    shifted[:, 1:] = labels[:, :-1].clone()
    shifted[:, 0] = decoder_start_token_id
    shifted.masked_fill_(shifted == -100, pad_token_id)
    return shifted


def embedding_aux_loss(logits: torch.Tensor,
                       labels: torch.Tensor,
                       token_embedding_weight: torch.Tensor,
                       pad_token_id: int,
                       decoder_start_token_id: int,
                       lambda_embed: float,
                       loss_type: str = "l2") -> torch.Tensor:
    """
    logits: [batch, seq, vocab]
    labels: [batch, seq]
    token_embedding_weight: [vocab, hidden]
    """
    # CE over labels
    ce_loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )

    # Build decoder inputs so positions align with labels
    decoder_input_ids = shift_right(labels, pad_token_id, decoder_start_token_id)

    # Predicted distribution over vocab
    probs = torch.softmax(logits, dim=-1)  # [B, T, V]

    # Softmax-weighted average embedding at each position
    e_hat = probs @ token_embedding_weight  # [B, T, H]

    # Ground-truth token embedding at each position
    gt_ids = labels.clone()
    gt_ids = gt_ids.masked_fill(gt_ids == -100, pad_token_id)
    e_gt = token_embedding_weight[gt_ids]  # [B, T, H]

    # Mask padding positions
    mask = (labels != -100).unsqueeze(-1)

    if loss_type == "l2":
        per_token = (e_hat - e_gt).pow(2).sum(dim=-1)  # [B, T]
    elif loss_type == "cosine":
        per_token = 1.0 - F.cosine_similarity(e_hat, e_gt, dim=-1)  # [B, T]
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    embed_loss = (per_token * mask.squeeze(-1)).sum() / mask.sum().clamp_min(1)

    return ce_loss + lambda_embed * embed_loss, {
        "ce_loss": ce_loss.detach(),
        "embed_loss": embed_loss.detach(),
    }

In [15]:
# -------------------------
# 3) Trainer wrapper
# -------------------------
class EmbeddingRegSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(self, *args, lambda_embed=0.1, loss_type="l2", **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_embed = lambda_embed
        self.loss_type = loss_type

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        logits = outputs.logits

        token_embedding_weight = model.get_input_embeddings().weight
        loss, aux = embedding_aux_loss(
            logits=logits,
            labels=labels,
            token_embedding_weight=token_embedding_weight,
            pad_token_id=self.tokenizer.pad_token_id,
            decoder_start_token_id=model.config.decoder_start_token_id,
            lambda_embed=self.lambda_embed,
            loss_type=self.loss_type,
        )

        if return_outputs:
            outputs.loss = loss
            outputs.aux_losses = aux
            return loss, outputs
        return loss

In [16]:
# -------------------------
# 4) Metrics
# -------------------------
def decode_predictions(pred_ids, tokenizer):
    return tokenizer.batch_decode(pred_ids, skip_special_tokens=True)


def compute_metrics_factory(tokenizer, comet_model=None, use_bertscore=False):
    bleu_metric = BLEU()
    chrf_metric = CHRF(word_order=2)  # chrF++

    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple):
            preds = preds[0]

        # Replace -100 in labels before decoding
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        pred_text = decode_predictions(preds, tokenizer)
        ref_text = decode_predictions(labels, tokenizer)

        # SacreBLEU / chrF++
        bleu = bleu_metric.corpus_score(pred_text, [ref_text]).score
        chrf = chrf_metric.corpus_score(pred_text, [ref_text]).score

        metrics = {
            "bleu": bleu,
            "chrfpp": chrf,
        }

        # Optional BERTScore
        if use_bertscore:
            P, R, F1 = bertscore_score(
                pred_text,
                ref_text,
                lang="en",  # target language here is English, because outputs are English
                rescale_with_baseline=True,
            )
            metrics["bertscore_f1"] = float(F1.mean().item())

        # COMET typically runs outside the Hugging Face compute_metrics loop
        return metrics

    return compute_metrics


def evaluate_with_comet(src_texts, pred_texts, ref_texts, comet_model):
    data = [
        {"src": s, "mt": mt, "ref": r}
        for s, mt, r in zip(src_texts, pred_texts, ref_texts)
    ]
    output = comet_model.predict(data, batch_size=8, gpus=1)
    return {
        "comet_system": float(output.system_score),
        "comet_sentence_scores": output.scores,
    }

In [17]:
# -------------------------
# 5) Train / eval loop
# -------------------------
def train_one_run(train_ds, val_ds, lambda_embed, loss_type="l2", output_dir="outputs"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(output_dir, f"lambda_{lambda_embed}_{loss_type}"),
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        predict_with_generate=True,
        generation_max_length=128,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    )

    trainer = EmbeddingRegSeq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        tokenizer=tokenizer,
        lambda_embed=lambda_embed,
        loss_type=loss_type,
        compute_metrics=compute_metrics_factory(tokenizer, use_bertscore=False),
    )

    trainer.train()
    eval_metrics = trainer.evaluate()

    return trainer, tokenizer, model, eval_metrics

In [25]:
# 1) Load data
train_hf, val_hf, test_hf = load_translation_data()
print("loaded dataset")
print(train_hf)
print(train_hf[0])
print(val_hf[0])
print(test_hf[0])

loaded dataset
Dataset({
    features: ['id', 'translation'],
    num_rows: 74776
})
{'id': '35571', 'translation': {'en': 'At first he thought that he was the police, but soon he found that he had some lay of his own.', 'es': 'Al principio mi cuñado pensó que era de la policía, pero pronto comprendió que trabaja por su cuenta.'}}
{'id': '46429', 'translation': {'en': '"Good," said Milady to herself; "without thinking what it is, he calls it a crime!"', 'es': '«Bueno dijo Milady para sus adentros , ¡sin saber lo que es, lo llama crimen!»'}}
{'id': '15944', 'translation': {'en': '"Such, in brief, were the words of the letter, words that made me set out at once without waiting any longer for reply or money; for I now saw clearly that it was not the purchase of horses but of his own pleasure that had made Don Fernando send me to his brother.', 'es': '»Éstas, en suma, fueron las razones que la carta contenía y las que me hicieron poner luego en camino, sin esperar otra respuesta ni otros d

In [26]:
# 2) Tokenizer / dataset wrappers
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = TranslationDataset(train_hf, tokenizer)
val_ds = TranslationDataset(val_hf, tokenizer)
test_ds = TranslationDataset(test_hf, tokenizer)

c:\Users\KatherynZhou\AppData\Local\anaconda3\envs\translation\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
# 3) Sweep regularization strength
lambda_values = [0.0, 0.01, 0.05, 0.1, 0.5]
results = []

comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

for lam in lambda_values:
    trainer, tokenizer, model, val_metrics = train_one_run(
        train_ds=None,   # replace with train_ds
        val_ds=None,     # replace with val_ds
        lambda_embed=lam,
        loss_type="l2",
    )

    # 4) Generate predictions on test set
    # preds = trainer.predict(test_ds)
    # pred_ids = preds.predictions.argmax(-1)  # or use generated tokens from predict_with_generate
    # pred_text = decode_predictions(pred_ids, tokenizer)

    # 5) Compute COMET separately from src / pred / ref
    # src_texts = ...
    # ref_texts = ...
    # comet_scores = evaluate_with_comet(src_texts, pred_text, ref_texts, comet_model)

    results.append({
        "lambda": lam,
        **val_metrics,
        # **comet_scores,
    })

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

c:\Users\KatherynZhou\AppData\Local\anaconda3\envs\translation\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KatherynZhou\.cache\huggingface\hub\datasets--Helsinki-NLP--opus_books. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not ins

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`